**Importing Libraries**

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import csv


**Accessing Links**

In [ ]:
def separate_link():
    links = []
    with open('car_links.csv', newline='') as file:
        reader = csv.reader(file)
        header = next(reader) #skip header
        links = [row[0] for row in reader if row] #make sure rows aren't empty
    return links

**Scraping Unique Ads,** with the help of debugging techniques (logging)

In [ ]:
def scrape_ad_details(response):
    """
    Scrapes ad details from a car listing page
    Returns a dictionary with extracted information
    """
    soup = BeautifulSoup(response.text, 'lxml')
    info = {}
    
    try:
        # Add debug info about the page structure
        print(f"Page title: {soup.title.string if soup.title else 'No title found'}")
        
        # Find the main details container
        details = soup.find('div', class_='box d-none d-md-block') 
        if not details:
            print("Error: Main details container not found")
            print("Available divs with 'box' class:")
            box_divs = soup.find_all('div', class_=lambda x: x and 'box' in x) #discover what the website is actually using instead of what I am expecting
            for i, div in enumerate(box_divs[:5]):  # Show first 5 matches
                print(f"  {i}: {div.get('class')}")
            return None
        
        # Extract title with better error handling
        title_element = soup.find('h1', class_='d-none d-md-block occasion-title')
        if not title_element:
            # Try alternative selectors
            title_element = soup.find('h1', class_='occasion-title') or soup.find('h1')
        info['title'] = title_element.get_text(strip=True) if title_element else ''
        
        # Extract price with better error handling
        price_element = details.find('div', class_='price')
        info['price'] = price_element.get_text(strip=True) if price_element else ''
        
        # Find main specs section
        main_specs = details.find('div', class_='main-specs')
        if not main_specs:
            print("Error: main-specs div not found")
            return None
            
        main_specs_list = main_specs.find_all('li')
        print(f"Found {len(main_specs_list)} main specs items")
        
        # Debug: Print all main specs
        for i, li in enumerate(main_specs_list):
            print(f"Main spec {i}: {li.get_text(strip=True)}")
        
        # Handle brand and model extraction more robustly
        brand_div = soup.find('div', class_='col-md-6 mb-3 mb-md-0')
        if brand_div:
            brand_items = brand_div.find_all('li')
            print(f"Found {len(brand_items)} brand items")
            
            # Extract brand
            if len(brand_items) > 0:
                brand_span = brand_items[0].find('span', class_='spec-value text-end')
                if brand_span:
                    brand_tag = brand_span.find('a')
                    info['brand'] = brand_tag.get_text(strip=True) if brand_tag else brand_span.get_text(strip=True)
                else:
                    info['brand'] = ''
            else:
                info['brand'] = ''
                
            # Extract model
            if len(brand_items) > 1:
                model_span = brand_items[1].find('span', class_='spec-value text-end')
                if model_span:
                    model_tag = model_span.find('a')
                    info['model'] = model_tag.get_text(strip=True) if model_tag else model_span.get_text(strip=True)
                else:
                    info['model'] = ''
            else:
                info['model'] = ''
            
            #Extract Interior
            if len(brand_items) > 2:
                interior_span = brand_items[6].find('span', class_='spec-value text-end')
                if interior_span:
                    interior_tag = interior_span.find('a')
                    info['interior'] = interior_tag.get_text(strip=True) if interior_tag else interior_span.get_text(strip=True)
                else:
                    info['interior'] = ''
            else:
                info['interior'] = ''

        else:
            info['brand'] = ''
            info['model'] = ''
            info['interior'] = ''
        
        # Extract main specs with bounds checking, handle every case without crashing
        spec_fields = [
            ('mileage', 0),
            ('circulation-date', 1),
            ('fuel', 2),
            ('gear', 3),
            ('fiscal-power', 4),
            ('body-type', 6),
            ('ownership', 8),
            ('publish-date', 9),
            ('location', 10)
        ]
        
        for field_name, index in spec_fields:
            if index < len(main_specs_list):
                spec_element = main_specs_list[index].find('span', class_='spec-value')
                info[field_name] = spec_element.get_text(strip=True) if spec_element else ''
            else:
                info[field_name] = ''
                print(f"Warning: Index {index} not available for {field_name}")
        
        # Extract engine size
        # Target the div that doesn't have the additional 'mb-3 mb-md-0' classes
        col_divs = soup.find_all('div', class_='col-md-6')
        engine_div = None
        for div in col_divs:
            if div.find('div', class_='divided-specs'):
                if 'mb-3' not in div.get('class', []):
                    engine_div = div
                    break
        if engine_div:
            divided_specs = engine_div.find('div', class_='divided-specs')
            if divided_specs:
                engine_items = divided_specs.find_all('li')
                print(f"Found {len(engine_items)} engine items")
                
                if len(engine_items) > 6:
                    engine_span = engine_items[6].find('span', class_='spec-value text-end')
                    info['engine-size'] = engine_span.get_text(strip=True) if engine_span else ''
                else:
                    info['engine-size'] = ''
            else:
                info['engine-size'] = ''
        else:
            info['engine-size'] = ''
        
        # Extract description
        desc_div = soup.find('div', class_='col-md-8')
        if desc_div:
            desc_sub_div = desc_div.find('div', class_='clamped-text')
            if desc_sub_div:
                desc_text = desc_sub_div.find('p', class_='text')
                info['description'] = desc_text.get_text(strip=True) if desc_text else ''
            else:
                info['description'] = ''
        else:
            info['description'] = ''
        
        print(f"Successfully extracted info: {info}")
        return info
        
    except Exception as e:
        print(f'Error extracting ad info: {e}')
        print(f'Details found: {details is not None if "details" in locals() else "details variable not set"}')
        print(f'Main specs count: {len(main_specs_list) if "main_specs_list" in locals() else "main_specs_list not found"}')
        
        # Save HTML for debugging
        with open('debug_page.html', 'w', encoding='utf-8') as f:
            f.write(response.text)
        print("Saved HTML content to debug_page.html for inspection")
        
        return None

**DataFrame Creation ~ Data Storage**

In [ ]:
def df_creation():
    """
    Creates a DataFrame by scraping car listing details
    """
    links = separate_link()  
    num = 1
    df = pd.DataFrame(columns=[
        'title', 'price', 'brand', 'model', 'mileage', 'circulation-date',
        'fuel', 'engine-size', 'gear', 'fiscal-power', 'body-type',
        'ownership', 'publish-date', 'location', 'description', 'interior', 'link'
    ])
    
    successful_scrapes = 0
    
    for link in links:  # Testing with first link only
        print(f'\n{"="*50}')
        print(f'Scraping ad number: {num}')
        print(f'URL: {link}')
        print(f'{"="*50}')
        
        try:
            # Add headers to mimic a real browser
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            
            response = requests.get(link, timeout=10, headers=headers)
            response.raise_for_status()
            
            print(f"Response status: {response.status_code}")
            print(f"Content length: {len(response.text)} characters")
            
            ad_details = scrape_ad_details(response)
            
            if ad_details:
                df = pd.concat([df, pd.DataFrame([ad_details])], ignore_index=True)
                df['link'] = link
                successful_scrapes += 1
                print(f'✓ Successfully scraped ad number: {num}')
            else:
                print(f'✗ No data extracted from ad number: {num}')
                
        except requests.exceptions.RequestException as e:
            print(f'✗ Failed to scrape page number: {num}! Error: {e}')
            
        except Exception as e:
            print(f'✗ Unexpected error for ad {num}: {e}')
        
        num += 1
        time.sleep(random.uniform(1, 3))
    
    success_rate = round((successful_scrapes / len(links)) * 100, 2) if links else 0
    print(f'\n{"="*50}')
    print(f'FINAL RESULTS:')
    print(f'Total links: {len(links)}')
    print(f'Successful scrapes: {successful_scrapes}')
    print(f'Success Rate: {success_rate}%')
    print(f'{"="*50}')
    
    return df

**Error Pages Debugging, to use when scraper stops working**

In [ ]:
def debug_single_page(url):
    """
    Debug a single page to understand its structure
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'lxml')
        
        print("=== PAGE STRUCTURE DEBUG ===")
        print(f"Title: {soup.title.string if soup.title else 'No title'}")
        print(f"Total divs: {len(soup.find_all('div'))}")
        
        # Check for main container
        main_container = soup.find('div', class_='box d-none d-md-block')
        print(f"Main container found: {main_container is not None}")
        
        if not main_container:
            print("Alternative containers:")
            for div in soup.find_all('div', class_=lambda x: x and 'box' in x)[:5]:
                print(f"  - {div.get('class')}")
        
        # Check for specs
        specs_containers = soup.find_all('div', class_='main-specs')
        print(f"Specs containers found: {len(specs_containers)}")
        
        return response
        
    except Exception as e:
        print(f"Debug error: {e}")
        return None

**Code Execution & CSV File Creation**

In [ ]:
df = df_creation()
#91.05% success rate

In [ ]:
df.to_csv('first_set_v2.csv', index=False)